# Bab 7: Membina Aplikasi Sembang
## Permulaan Pantas Microsoft Foundry Models API

Buku nota ini diubah suai daripada [Repositori Contoh Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) yang merangkumi buku nota yang mengakses perkhidmatan [Azure OpenAI](aoai-assignment.ipynb).

> **Nota:** GitHub Models akan dihentikan pada akhir Julai 2026. Buku nota ini kini menyasarkan [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), yang menawarkan katalog model percuma untuk dicuba dan pengalaman SDK Azure AI Inference yang sama.


# Gambaran Keseluruhan  
"Model bahasa besar adalah fungsi yang memetakan teks kepada teks. Diberikan rentetan teks input, model bahasa besar cuba meramalkan teks yang akan datang seterusnya"(1). Notebook "quickstart" ini akan memperkenalkan pengguna kepada konsep LLM peringkat tinggi, keperluan pakej teras untuk memulakan dengan AML, pengenalan ringkas kepada reka bentuk arahan, dan beberapa contoh ringkas bagi pelbagai kes penggunaan. 


## Jadual Kandungan  

[Gambaran Keseluruhan](#overview)  
[Cara Menggunakan Perkhidmatan OpenAI](#how-to-use-openai-service)  
[1. Mewujudkan Perkhidmatan OpenAI Anda](#1.-creating-your-openai-service)  
[2. Pemasangan](#2.-installation)    
[3. Kredensial](#3.-credentials)  

[Kes Penggunaan](#use-cases)    
[1. Meringkaskan Teks](#1.-summarize-text)  
[2. Mengklasifikasikan Teks](#2.-classify-text)  
[3. Menghasilkan Nama Produk Baharu](#3.-generate-new-product-names)  
[4. Melatih Semula Pengklasifikasi](#4.fine-tune-a-classifier)  

[Rujukan](#references)


### Bina arahan pertama anda  
Latihan ringkas ini akan memberikan pengenalan asas untuk menghantar arahan ke model dalam Microsoft Foundry Models untuk tugasan ringkas "penyimpulan".


**Langkah-langkah**:  
1. Pasang perpustakaan `azure-ai-inference` dalam persekitaran python anda, jika anda belum melakukannya.  
2. Muatkan perpustakaan pembantu standard dan tetapkan kelayakan Microsoft Foundry Models anda.  
3. Pilih model untuk tugasan anda  
4. Cipta arahan ringkas untuk model tersebut  
5. Hantar permintaan anda ke API model!


### 1. Pasang `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Import perpustakaan pembantu dan cipta kelayakan


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Mencari model yang sesuai  
Model seperti GPT-4o dan GPT-4o mini boleh memahami dan menjana bahasa semula jadi, dan tersedia dalam katalog Model Microsoft Foundry bersama-sama model dari Meta, Mistral, Cohere, dan Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. Reka Bentuk Prompt  

"Keajaiban model bahasa besar adalah dengan dilatih untuk meminimumkan ralat ramalan ini ke atas sejumlah besar teks, model akhirnya mempelajari konsep yang berguna untuk ramalan ini. Contohnya, mereka mempelajari konsep seperti"(1):

* cara mengeja
* cara tatabahasa berfungsi
* cara dan rephrasing
* cara menjawab soalan
* cara berkomunikasi dalam perbualan
* cara menulis dalam pelbagai bahasa
* cara menulis kod
* dan sebagainya.

#### Cara mengawal model bahasa besar  
"Daripada semua input kepada model bahasa besar, yang paling berpengaruh adalah prompt teks(1).

Model bahasa besar boleh diprompt untuk menghasilkan output dalam beberapa cara:

Arahan: Beritahu model apa yang anda mahu
Lengkapkan: Membuat model melengkapkan permulaan apa yang anda mahu
Demonstrasi: Tunjukkan kepada model apa yang anda mahu, dengan sama ada:
Beberapa contoh dalam prompt
Beratus-ratus atau ribuan contoh dalam dataset latihan penalaan halus"



#### Terdapat tiga garis panduan asas untuk membuat prompt:

**Tunjuk dan beritahu**. Jelaskan apa yang anda mahu sama ada melalui arahan, contoh, atau gabungan kedua-duanya. Jika anda mahu model menyusun senarai item mengikut turutan abjad atau mengklasifikasikan perenggan mengikut sentimen, tunjukkan ia itu yang anda mahukan.

**Sediakan data berkualiti**. Jika anda cuba membina pengklasifikasi atau mengarahkan model mengikuti corak, pastikan terdapat cukup contoh. Pastikan anda menyemak contoh anda — model biasanya cukup bijak untuk mengesan kesilapan ejaan asas dan memberi anda respons, tetapi ia juga mungkin menganggap ini disengajakan dan ia boleh menjejaskan respons.

**Semak tetapan anda.** Tetapan suhu dan top_p mengawal betapa deterministik model dalam menghasilkan respons. Jika anda meminta respons di mana hanya ada satu jawapan yang betul, maka anda ingin menetapkan ini lebih rendah. Jika anda mahukan respons yang lebih pelbagai, maka anda mungkin mahu menetapkannya lebih tinggi. Kesilapan nombor satu yang dilakukan orang dengan tetapan ini ialah menganggap bahawa ia adalah kawalan "kepintaran" atau "kreativiti".


Sumber: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Hantar!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Ulangi panggilan yang sama, bagaimana perbandingan hasilnya?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Ringkaskan Teks  
#### Cabaran  
Ringkaskan teks dengan menambah 'tl;dr:' di akhir petikan teks. Perhatikan bagaimana model memahami cara melaksanakan beberapa tugas tanpa arahan tambahan. Anda boleh bereksperimen dengan arahan yang lebih deskriptif daripada tl;dr untuk mengubah tingkah laku model dan menyesuaikan ringkasan yang anda terima(3).  

Kerja terkini telah menunjukkan peningkatan ketara dalam banyak tugas dan penanda aras NLP dengan pra-latihan pada korpus teks yang besar diikuti dengan penalaan halus pada tugas tertentu. Walaupun biasanya bebas tugas dalam seni bina, kaedah ini masih memerlukan set data penalaan halus khusus tugas yang mengandungi ribuan atau puluhan ribu contoh. Sebaliknya, manusia biasanya boleh melaksanakan tugas bahasa baru hanya dari beberapa contoh atau dari arahan mudah - sesuatu yang masih sukar dilakukan oleh sistem NLP semasa. Di sini kami menunjukkan bahawa skala besar model bahasa meningkatkan prestasi bebas tugas, beberapa contoh, kadang-kadang mencapai tahap bersaing dengan pendekatan penalaan halus terkini yang terbaik.  



Tl;dr


# Latihan untuk beberapa kes penggunaan  
1. Meringkaskan Teks  
2. Mengklasifikasikan Teks  
3. Menghasilkan Nama Produk Baru


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Klasifikasikan Teks  
#### Cabaran  
Klasifikasikan item ke dalam kategori yang diberikan semasa masa inferens. Dalam contoh berikut, kami menyediakan kedua-dua kategori dan teks untuk diklasifikasikan dalam prompt(*playground_reference). 

Pertanyaan Pelanggan: Hai, salah satu kekunci pada papan kekunci komputer riba saya baru-baru ini rosak dan saya memerlukan pengganti:

Kategori yang diklasifikasikan:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Jana Nama Produk Baru
#### Cabaran
Cipta nama produk dari perkataan contoh. Di sini kami sertakan dalam arahan maklumat tentang produk yang kami akan jana nama. Kami juga menyediakan contoh serupa untuk menunjukkan corak yang kami ingin terima. Kami juga telah menetapkan nilai suhu tinggi untuk meningkatkan kebarangkalian dan respons yang lebih inovatif.

Penerangan produk: Pembuat milkshake rumah
Perkataan benih: laju, sihat, padat.
Nama produk: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Penerangan produk: Sepasang kasut yang boleh muat pada sebarang saiz kaki.
Perkataan benih: boleh disesuaikan, muat, omni-fitting.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Rujukan  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Portal Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Amalan terbaik untuk menyesuaikan model GPT bagi mengklasifikasikan teks](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Untuk Bantuan Lanjut  
[Pasukan Komersialisasi OpenAI](AzureOpenAITeam@microsoft.com) 


# Penyumbang
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan oleh manusia profesional adalah disyorkan. Kami tidak bertanggungjawab terhadap sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
